# Explainable Skin Cancer Detection Using CNN (HAM10000)
## Google Colab GPU-Accelerated Training Notebook

This notebook trains a **MobileNetV2** transfer learning classifier on the complete **HAM10000** dataset (10,015 dermoscopic images) across all 7 diagnostic categories, with **lesion-group leakage prevention**, **class imbalance weighting**, and **Grad-CAM explainability**.

### Recommended Colab Settings:
1. Go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or A100/V100 if available) as the Hardware accelerator.
3. Run all cells sequentially.

In [ ]:
# Verify GPU Hardware Acceleration
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[PASS] GPU detected: {gpus}")
else:
    print("[WARNING] No GPU detected. Please go to Runtime -> Change runtime type -> select T4 GPU.")

### Step 1: Install Dependencies and Download HAM10000 Dataset
You can easily download the dataset using Kaggle API or from Google Drive.

In [ ]:
!pip install -q kaggle scikit-learn seaborn

import os
# Create directories
os.makedirs("data/HAM10000", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Check if kaggle.json is uploaded to download automatically:
if os.path.exists("kaggle.json"):
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/HAM10000 --unzip
else:
    print("Note: Upload 'kaggle.json' to Colab root to download directly, or upload the HAM10000 zip files to data/HAM10000/")

### Step 2: Canonical 7-Class Dataset Discovery & Leakage Prevention

In [ ]:
import glob, json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

CANONICAL_CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CANONICAL_CLASSES)}

DATA_DIR = "data/HAM10000"
csv_candidates = glob.glob(os.path.join(DATA_DIR, "**", "*metadata*.csv"), recursive=True)
if not csv_candidates:
    raise FileNotFoundError("Metadata CSV not found under data/HAM10000/")
metadata_path = csv_candidates[0]
print(f"Using metadata: {metadata_path}")

df = pd.read_csv(metadata_path)
image_map = {}
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in glob.glob(os.path.join(DATA_DIR, "**", ext), recursive=True):
        img_id = os.path.splitext(os.path.basename(p))[0]
        image_map[img_id] = os.path.abspath(p)

df["path"] = df["image_id"].map(image_map)
df = df.dropna(subset=["path", "dx"]).reset_index(drop=True)
df["label"] = df["dx"].map(CLASS_TO_IDX).astype(int)
print(f"Total matched images: {len(df)}")
print("Class distribution:")
print(df["dx"].value_counts())

# Prevent lesion leakage across splits using GroupShuffleSplit on lesion_id
groups = df["lesion_id"].astype(str) if "lesion_id" in df.columns else df["image_id"].astype(str)
split1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(split1.split(df, df["label"], groups))

trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

groups_trainval = trainval_df["lesion_id"].astype(str) if "lesion_id" in trainval_df.columns else trainval_df["image_id"].astype(str)
split2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(split2.split(trainval_df, trainval_df["label"], groups_trainval))

train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)
print(f"Partitions: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Verify zero leakage
assert len(set(train_df["lesion_id"]).intersection(set(test_df["lesion_id"]))) == 0, "Leakage detected!"
print("[PASS] Zero data leakage verified between train and test groups.")

### Step 3: Dataset Pipelines with Augmentation and Class Weights

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=42),
    tf.keras.layers.RandomRotation(0.15, seed=42),
    tf.keras.layers.RandomZoom(0.1, seed=42)
])

def preprocess_img(path, label, is_training=False):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    if is_training:
        img = augmenter(tf.expand_dims(img, 0))[0]
    return img, tf.one_hot(label, depth=7)

def build_tf_ds(frame, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["label"].values))
    if is_training:
        ds = ds.shuffle(min(len(frame), 3000), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l: preprocess_img(p, l, is_training), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = build_tf_ds(train_df, True)
val_ds = build_tf_ds(val_df, False)
test_ds = build_tf_ds(test_df, False)

# Class weights to balance dataset distribution
weights = compute_class_weight('balanced', classes=np.unique(train_df["label"]), y=train_df["label"].values)
class_weights = {int(cls): float(w) for cls, w in zip(np.unique(train_df["label"]), weights)}
print("Computed class weights:", class_weights)

### Step 4: Transfer Learning Model Definition & Training

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(7, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="ham10000_mobilenetv2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("models/skin_lesion_model.keras", monitor="val_loss", save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weights,
    callbacks=callbacks
)

with open("models/classes.json", "w") as f:
    json.dump(CANONICAL_CLASSES, f, indent=2)

### Step 5: Test Evaluation and Artifact Packaging

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

best_model = tf.keras.models.load_model("models/skin_lesion_model.keras")
y_true, y_pred = [], []

for imgs, labels in test_ds:
    preds = best_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

acc = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=CANONICAL_CLASSES, output_dict=True, zero_division=0)
print(f"Held-Out Test Accuracy: {acc * 100:.2f}%")

with open("models/evaluation_report.json", "w") as f:
    json.dump({"accuracy": float(acc), "sample_count": len(y_true), "classification_report": report}, f, indent=2)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CANONICAL_CLASSES, yticklabels=CANONICAL_CLASSES, cmap="Blues")
plt.title("Test Confusion Matrix")
plt.savefig("models/confusion_matrix.png", dpi=180)
plt.show()

# Package artifacts for download to local application
!zip -r trained_artifacts.zip models/ results/
print("Trained artifacts zipped. Ready to download!")
try:
    from google.colab import files
    files.download('trained_artifacts.zip')
except Exception:
    pass